### Import

In [1]:
import pinocchio as pin  ##type: ignore
from pinocchio.visualize import MeshcatVisualizer  ##type: ignore
import numpy as np
import sys
from inverse_kinematics import solve_ik
from scipy.stats import qmc
from scipy.sparse import save_npz, load_npz

from prm_sampling import sample_batch, sample_valid_X_parallel
from prm_graph import build_q_pool
from self_collision import build_link_capsules, check_self_collision, default_excluded_pairs, filter_internal_collision_free_parallel
from prm_edges import interpolate_q, edge_internal_collision_free
from prm_roadmap import build_prm_graph_parallel
from prm_query import embed_X_in_graph, dijkstra_path
from workspace import is_pose_reachable
from simulators import simulate_lqr_from_path

### Load model

In [2]:
model, collision_model, visual_model = pin.buildModelsFromUrdf(
    "lbr_iiwa7_r800.urdf", package_dirs="."
)
data = model.createData()

### Initialize visualizer (meshcat)

In [3]:
try:
    viz = MeshcatVisualizer(model, collision_model, visual_model)
    viz.initViewer(open=True)
    viz.loadViewerModel()
except ImportError as err:
    print(
        "Error while initializing the viewer. "
        "It seems you should install Python meshcat"
    )
    print(err)
    sys.exit(0)

You can open the visualizer by visiting the following URL:
http://127.0.0.1:7006/static/


### Create X samples

In [ ]:
n_samples = int(1e6)

positions, rotations, q_solutions, num_valid = sample_valid_X_parallel(
    n_samples, model, "lbr_iiwa7_r800.urdf", ".", rng=0
)
print(f"{num_valid}/{n_samples} samples IK-verified valid")

### Save valid X (positions + orientations) to data/

In [ ]:
import os

os.makedirs("data", exist_ok=True)
np.savez_compressed(
    "data/valid_X.npz",
    positions=positions,
    rotations=rotations,
    q_solutions=q_solutions,
)
print(f"saved {len(positions)} valid (position, rotation, q) samples to data/valid_X.npz")

### Load saved X samples

In [ ]:
loaded = np.load("data/valid_X.npz")
positions, rotations = loaded["positions"], loaded["rotations"]
print(f"loaded {len(positions)} valid X samples from data/valid_X.npz")

### Build the q-pool (redundancy circle, m knot points, both elbow branches, per X)

In [ ]:
m = 20  # knot points in phi per elbow branch (per X) -- benchmarked middle ground: m=4 -> 1.63M nodes/~124s, m=12 -> 4.9M nodes/~379s

q_pool = build_q_pool(positions, rotations, model, m=m)
print(f"built q pool: {q_pool.shape[0]} nodes from {len(positions)} valid X (m={m}, elbow_signs=(1,-1))")

### Save q pool to data/

In [ ]:
np.savez_compressed("data/q_pool.npz", q_pool=q_pool)
print(f"saved {q_pool.shape[0]} q's to data/q_pool.npz")

### Load saved q pool

In [4]:
loaded = np.load("data/q_pool.npz")
q_pool = loaded["q_pool"]
print(f"loaded {q_pool.shape[0]} q pool nodes from data/q_pool.npz")

loaded 8162400 q pool nodes from data/q_pool.npz


### Filter q pool for internal (self-)collision-free configs

In [ ]:
from self_collision import filter_internal_collision_free_parallel

q_pool_internal_collision_free, n_internal_collision_free = filter_internal_collision_free_parallel(
    q_pool, "lbr_iiwa7_r800.urdf", "."
)
print(f"{n_internal_collision_free}/{q_pool.shape[0]} q's are internal-collision-free")

### Save internal-collision-free q pool to data/

In [ ]:
np.savez_compressed("data/q_pool_internal_collision_free.npz", q_pool=q_pool_internal_collision_free)
print(f"saved {q_pool_internal_collision_free.shape[0]} internal-collision-free q's to data/q_pool_internal_collision_free.npz")

### Load saved internal-collision-free q pool

In [5]:
loaded = np.load("data/q_pool_internal_collision_free.npz")
q_pool_internal_collision_free = loaded["q_pool"]
print(f"loaded {q_pool_internal_collision_free.shape[0]} internal-collision-free q pool nodes from data/q_pool_internal_collision_free.npz")

loaded 3728688 internal-collision-free q pool nodes from data/q_pool_internal_collision_free.npz


### Build PRM graph (k-NN candidate edges, internal-collision-free only, parallel)

In [ ]:
k = 10
prm_graph = build_prm_graph_parallel(q_pool_internal_collision_free, "lbr_iiwa7_r800.urdf", ".", k=k)
print(f"built PRM graph: {prm_graph.shape[0]} nodes, {prm_graph.nnz // 2} edges (k={k})")

In [ ]:
from scipy.sparse import save_npz

save_npz("data/prm_graph.npz", prm_graph)
print(f"saved PRM graph ({prm_graph.shape[0]} nodes, {prm_graph.nnz // 2} edges) to data/prm_graph.npz")

### Load saved PRM graph

In [6]:
prm_graph = load_npz("data/prm_graph.npz")
print(f"loaded PRM graph: {prm_graph.shape[0]} nodes, {prm_graph.nnz // 2} edges from data/prm_graph.npz")

loaded PRM graph: 3728688 nodes, 23133422 edges from data/prm_graph.npz


### Random start/goal X -> embed -> Dijkstra -> spline -> TVLQR -> simulate

In [8]:
rng = np.random.default_rng()
capsules = build_link_capsules(collision_model)
excluded_pairs = default_excluded_pairs(len(capsules))

def random_valid_X():
    while True:
        xyz, R, n_acc = sample_batch(1, model, rng=rng)
        if n_acc and is_pose_reachable(model, data, xyz[0], R[0]):
            return xyz[0], R[0]

X_pair = [random_valid_X(), random_valid_X()]
aug_graph, aug_q_pool, node_idxs = embed_X_in_graph(
    X_pair, prm_graph, q_pool_internal_collision_free, model, data, capsules, excluded_pairs
)
while not node_idxs[0] or not node_idxs[1]:
    if not node_idxs[0]:
        X_pair[0] = random_valid_X()
    if not node_idxs[1]:
        X_pair[1] = random_valid_X()
    aug_graph, aug_q_pool, node_idxs = embed_X_in_graph(
        X_pair, prm_graph, q_pool_internal_collision_free, model, data, capsules, excluded_pairs
    )

start_idx, goal_idx = node_idxs[0][0], node_idxs[1][0]
path, dist = dijkstra_path(aug_graph, start_idx, goal_idx)
print(f"start X: {X_pair[0][0]}, goal X: {X_pair[1][0]}")
print(f"dijkstra path: {len(path)} nodes, distance {dist:.4f}")

use_shortcutting = True
q_path = aug_q_pool[path]
result = simulate_lqr_from_path(
    model, q_path, T=len(path) * 0.5, dt=0.01, viz=viz, viz_every=5, real_time=True,
    use_shortcutting=use_shortcutting, capsules=capsules, excluded_pairs=excluded_pairs,
)
print(f"simulated {len(result['t'])} steps, max tracking error {result['errors'].max():.4f}, final error {result['errors'][-1]:.4f}")

start X: [-0.15725565  0.32678043  0.60159249], goal X: [-0.39727425  0.41748835  0.51253289]
dijkstra path: 20 nodes, distance 8.4681
simulated 1001 steps, max tracking error 0.0056, final error 0.0054
